<a href="https://colab.research.google.com/github/Nyangweso1/cybershujaa-data-ai-track/blob/main/Week1_Assignment1_Web_Scraping_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1, Assignment #1: Web Data Scraping

**Course:** Data & AI Specialist Program, Cyber Shujaa
**Opens:** Monday, 7 September 2026
**Due:** Sunday, 13 September 2026, 23:59
**Submit:** this notebook (.ipynb) plus your exported .csv file, via the file submission area on the LMS

## What this assignment is actually testing

In class we walked through the Data Science Methodology. This assignment is your first hands-on
pass through two of its steps: **Data Understanding** (going and getting real data instead of
being handed a clean file) and **Data Preparation** (fixing up what you collected before it is
usable). A live website is messier than any textbook dataset, which is exactly the point.

## Your target site (fixed, not a free choice)

Every student scrapes the same page for this assignment:
[scrapethissite.com/pages/simple/](https://www.scrapethissite.com/pages/simple/), "Countries of
the World: A Simple Example." It lists all 250 countries in the world, each with a capital,
population, and area. Using one fixed page keeps the assignment fair to grade and makes sure
everyone hits the same real-world snag: this page is **not** built as a `<table>`, unlike the
hockey site from the in-class demo, so you cannot reuse that code unchanged. You will apply the
same four-step pattern (fetch, parse, find the repeating structure, collect into a DataFrame) to
HTML shaped differently, which is the actual skill this assignment is testing.

## How this notebook works

This is a **scaffold, not a solution**. Some cells run as they are. Others have a `# TODO`
comment where you need to write real code yourself, using what you learned in the in-class demo
plus your own investigation of this page's HTML in DevTools. If a cell runs without errors but
you never replaced a `___` placeholder, you have not actually finished that step.

## Academic honesty

Everyone is scraping the same page, so it will be tempting to just copy a classmate's finished
selectors. Don't, the reflection questions at the end ask you to explain choices you made while
writing this code, and a script you did not actually write yourself will not let you answer them
honestly. Find the tags and classes below through your own inspection of the page.


\

## Step 1: Import your libraries

These three libraries do three different jobs in this pipeline: `requests` talks to the
website over HTTP, `BeautifulSoup` (from the `bs4` package) turns raw HTML text into something
you can search through, and `pandas` gives you the DataFrame you will store your results in.


In [ ]:
# Colab already has these installed, but this line is harmless if they are already present
# and saves you a confusing NameError if you are running this outside Colab.
!pip install requests beautifulsoup4 pandas --quiet

import requests
from bs4 import BeautifulSoup
import pandas as pd


## Step 2: Inspect your target page

Your target is fixed: `https://www.scrapethissite.com/pages/simple/`. Before writing any
scraping code, check whether scraping is allowed: visit
`https://www.scrapethissite.com/robots.txt` in your browser and read it.

**Do this now, before running the next cell:**
1. Open the target page in a normal browser tab.
2. Right-click on one of the 250 country blocks and choose "Inspect" (or press F12).
3. Find the HTML tag and class name that wraps **one single country** (not the whole page, not
   just the country's name). You will need this exact tag and class name in Step 5.
4. While you are in there, also find the tags used for the capital, population, and area inside
   one country block. You will need those in Step 6.


In [ ]:
url = "https://www.scrapethissite.com/pages/simple/"  # fixed target, same for everyone

# I looked at the HTML layout of the target site with DevTools.

# One country is inside a div that has the class 'country'.

# The country name is inside an h3 element that has the class 'country-name'.

# The capital is inside a span that has the class 'country-capital'.

# The population is inside a span that has the class 'country-population'.

# The area is, inside a span that has the class 'country-area'.
robots_txt_notes = "Disallow: /lessons/ and /faq/ only. /pages/ is not listed as disallowed, so scraping /pages/simple/ is allowed."

print(f"Target URL: {url}")
print(f"robots.txt check: {robots_txt_notes}")

Target URL: https://www.scrapethissite.com/pages/simple/
robots.txt check: Disallow: /lessons/ and /faq/ only. /pages/ is not listed as disallowed, so scraping /pages/simple/ is allowed.


## Step 3: Fetch the page

`requests.get()` does not throw an error just because a page returns a 404 or a 500. It happily
hands you back a broken response and lets you find out the hard way three cells later when
`soup.find()` returns `None`. Catch that here instead.


In [ ]:
response = requests.get(url)

# I used requests.get() to send a request to the website and get its HTML content.

# I checked the response status to confirm the page was retrieved successfully.
try:
    response.raise_for_status()
except requests.exceptions.HTTPError as err:
    print(f"Request failed with status {response.status_code}: {err}")
    raise
 # Execution continues even though it failed with a 404!, this is to why response.raise_for_status() is used to help with execution.


print(f"Status code: {response.status_code}")
print(f"Bytes received: {len(response.text)}")


Status code: 200
Bytes received: 203338


## Step 4: Parse the HTML

`response.text` is one enormous string. `BeautifulSoup` turns it into a tree you can actually
search through by tag and class, instead of doing string matching yourself.


In [ ]:
soup = BeautifulSoup(response.text, "html.parser")
# Web readable HTML

print(soup.prettify()[:800])


<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <title>
   Countries of the World: A Simple Example | Scrape This Site | A public sandbox for learning web scraping
  </title>
  <link href="/static/images/scraper-icon.png" rel="icon" type="image/png"/>
  <meta content="width=device-width, initial-scale=1.0" name="viewport"/>
  <meta content="A single page that lists information about all the countries in the world. Good for those just get started with web scraping." name="description"/>
  <link crossorigin="anonymous" href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/css/bootstrap.min.css" integrity="sha256-MfvZlkHCEqatNoGiOXveE8FIwMzZg4W85qfrfIFBfYc= sha512-dTfge/zgoMYpP7QbHy4gWMEGsbsdZeCXz7irItjcC3sPUFtf0kuFbDz/ixG7ArTxmDjLXDmezHubeNikyKGVyQ==" rel="stylesheet"/>



## Step 5: Find every repeating item on the page

This is the main challenge in this assignment. Every country on the page is wrapped in the same
repeating tag and class, the one you found with DevTools in Step 2. `soup.find_all()` returns
every element matching that tag and class as a list, all 250 of them.


In [ ]:
# Every country on this page sits inside a <div class="col-md-4 country">

items = soup.find_all("div", class_="country")

print(f"Found {len(items)} items on the page.")



Found 250 items on the page.


## Step 6: Extract the fields you actually care about

For each country found in Step 5, pull out four fields: **name, capital, population, and
area**. This is where most of the real work in this assignment lives: the exact inner tags and
classes for each field are yours to find in DevTools, the same way you found the repeating
container in Step 2.


In [ ]:
scraped_rows = []

for item in items:
    # I just stripped whitespace from the indentation around <h3 class="country-name">tag.
    name = item.find("h3", class_="country-name").text.strip()

    # I target the span directly instead of the strong tag,
    # since the strong tag only holds the label text ("Capital:"), not the value I wanted.
    capital = item.find("span", class_="country-capital").text.strip()
    population = item.find("span", class_="country-population").text.strip()
    area_km2 = item.find("span", class_="country-area").text.strip()

    row = {
        "name": name,
        "capital": capital,
        "population": population,
        "area_km2": area_km2,
    }
    scraped_rows.append(row)


assert len(scraped_rows) > 0, "scraped_rows is still empty. Fill in the loop above before continuing."
print(f"Collected {len(scraped_rows)} rows.")


Collected 250 rows.


## Step 7: Build your DataFrame

You built a tiny version of this in class from a single value. Here you are doing the same
thing at real scale, from a whole list of dictionaries instead of one.


In [ ]:
df = pd.DataFrame(scraped_rows)

print(df.shape)
df.head()


(250, 4)


,name,capital,population,area_km2
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0


## Step 8: Clean your data (Data Preparation)

Real scraped data is rarely usable as-is. This is step three of the Data Science Methodology
from class, and it is not optional here. At minimum, check for and handle each of the
following. Not every issue will exist in your data, but you need to actually check, not assume.


In [ ]:

print("Missing values per column:")
print(df.isnull().sum())
#None appeared as missing some countries really have no capital listed (for example Antarctica). However the page writes the word "None" inside the span instead of leaving it empty so pandas reads it as the string "None" rather, than a real null value.


# checking for duplicate rows with df.duplicated().sum().
print(f"\nDuplicate rows: {df.duplicated().sum()}")
#no duplicate rows detected as of 250 countries in one page.

#checking the data types with df.dtypes.
print(f"\nDtypes before conversion:\n{df.dtypes}")
df["population"] = pd.to_numeric(df["population"], errors="coerce")
# Found that area_km2 has values written in scientific notation in the raw HTML (e.g., Russia's area is "1.71E7"),
df["area_km2"] = pd.to_numeric(df["area_km2"], errors="coerce")

print(df.dtypes)
df.head()


Missing values per column:
name          0
capital       0
population    0
area_km2      0
dtype: int64

Duplicate rows: 0

Dtypes before conversion:
name          object
capital       object
population    object
area_km2      object
dtype: object
name           object
capital        object
population      int64
area_km2      float64
dtype: object


,name,capital,population,area_km2
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0


## Step 9: Export to CSV

This is the deliverable file you submit alongside this notebook.


In [ ]:

output_filename = "assignment1_prudence_moraa_nyangweso.csv"
#used descriptive filename to make easy tracebility to my (.csv).

df.to_csv(output_filename, index=False)

# Proving that the export actually worked by reading the file back in, noticed no error is raised.

check_df = pd.read_csv(output_filename)
print(f"Saved and reloaded {check_df.shape[0]} rows, {check_df.shape[1]} columns.")
check_df.head()


Saved and reloaded 250 rows, 4 columns.


,name,capital,population,area_km2
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0


# DOWNLOADING "assignment1_prudence_moraa_nyangweso.csv" TO LOCAL MACHINE

In [ ]:
# downloaded my .csv file to my file explorer.
from google.colab import files
files.download('assignment1_prudence_moraa_nyangweso.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Step 10: Reflection (required, answer in this cell)

Replace the placeholders below with your own two or three sentence answers.

1. **What was the hardest part of getting your selectors right in Step 5 or Step 6, and how did
   you eventually figure it out?**
    The hardest part was finding the selectors for the repeating country blocks and the individual fields. I looked at the HTML structure that I saw while inspecting the page and tested tag and class names until I could pull the Country name, capital, population, and area are correct.

2. **What did Step 8 reveal about your data that you did not expect before you actually looked
   at it closely?**
   Step 8 revealed that there were no duplicate rows, but some values were stored as text even though they were numbers. I also saw that some countries had the capital listed as "None " and that some area values were written in notation. Therefore the data needed type conversion before analysis.

3. **If you had another hour, what would you add or fix in this scraper?**
  If I had another hour I would add data validation and error handling to make the scraper more reliable. I would also improve the cleaning process. I would add checks to make sure all scraped values are in the expected format before exporting the data to CSV.


## Submission checklist

- [ ] `robots_txt_notes` in Step 2 is filled in with your own real check, not a placeholder
- [ ] Step 3's status check actually stops on a failed request, not just prints a status code
- [ ] `items` in Step 5 is a non-empty list, roughly 250 items matching the page's own count
- [ ] `scraped_rows` in Step 6 has all four columns (name, capital, population, area) per row
- [ ] Step 8 shows evidence you actually checked for missing values, duplicates, and wrong types
- [ ] The exported .csv opens correctly when reloaded in Step 9
- [ ] All three reflection questions in Step 10 are answered in your own words
- [ ] You have submitted both this notebook (.ipynb) and your exported .csv file on the LMS

**Grading scale:** 0 = No submission, 1 = Below Expectation, 2 = Meets Expectation,
3 = Exceeds Expectation.